# 面试问题：多个模型之间怎样做质量、成本和延迟感知路由？

可直接复述的回答：路由目标不是永远选最大模型，而是在每个请求的质量底线、预算和延迟 SLO 下最大化效用。先建立同一黄金集上的模型质量、成本和延迟画像，再训练或手写请求难度估计。候选模型不满足硬约束时先剔除，再比较边际质量收益。路由评测要按请求配对，报告质量、花费、P95 延迟和失败切片。线上必须处理容量、熔断和画像漂移。探索流量需要受控，离线 IPS 只能在有 propensity 日志时使用。任何路由决策都要能解释。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：客服请求与模型 SLO 输入预览

六条脱敏请求保留难度、质量底线、成本上限和延迟 SLO。三个候选模型使用教学画像，表示小、中、大模型在不同难度下的质量与资源开销。


In [1]:
requests12 = [  # 构造带质量成本延迟约束的客服请求。
    {"id": "q1", "intent": "营业时间", "difficulty": 0.10, "min_quality": 0.65, "max_cost": 0.010, "sla_ms": 250},  # 简单事实问题适合小模型。
    {"id": "q2", "intent": "退款规则解释", "difficulty": 0.45, "min_quality": 0.74, "max_cost": 0.020, "sla_ms": 500},  # 中等规则解释。
    {"id": "q3", "intent": "多订单合并赔付", "difficulty": 0.85, "min_quality": 0.82, "max_cost": 0.060, "sla_ms": 1000},  # 高难请求允许大模型。
    {"id": "q4", "intent": "实时聊天短答", "difficulty": 0.30, "min_quality": 0.68, "max_cost": 0.008, "sla_ms": 200},  # 严格延迟和成本。
    {"id": "q5", "intent": "合同条款对比", "difficulty": 0.75, "min_quality": 0.79, "max_cost": 0.025, "sla_ms": 500},  # 预算限制下选择中模型。
    {"id": "q6", "intent": "高风险账户申诉", "difficulty": 0.95, "min_quality": 0.88, "max_cost": 0.060, "sla_ms": 1000},  # 需要最高质量或人工兜底。
]  # 完成六条真实路由请求。
models12 = {  # 定义三个候选模型的离线画像。
    "small": {"base_quality": 0.76, "difficulty_penalty": 0.22, "cost": 0.003, "latency_ms": 140},  # 小模型便宜快速但难题退化明显。
    "medium": {"base_quality": 0.87, "difficulty_penalty": 0.09, "cost": 0.014, "latency_ms": 360},  # 中模型提供平衡性能。
    "large": {"base_quality": 0.94, "difficulty_penalty": 0.03, "cost": 0.052, "latency_ms": 820},  # 大模型质量高但昂贵缓慢。
}  # 完成教学模型画像。
print("教学实验请求：id | intent | difficulty | quality/cost/SLA")  # 输出输入预览表头。
for request12 in requests12:  # 逐条展示路由约束。
    print(request12)  # 输出一条业务请求。
print("候选模型画像", models12)  # 展示路由器依赖的离线数据。


教学实验请求：id | intent | difficulty | quality/cost/SLA
{'id': 'q1', 'intent': '营业时间', 'difficulty': 0.1, 'min_quality': 0.65, 'max_cost': 0.01, 'sla_ms': 250}
{'id': 'q2', 'intent': '退款规则解释', 'difficulty': 0.45, 'min_quality': 0.74, 'max_cost': 0.02, 'sla_ms': 500}
{'id': 'q3', 'intent': '多订单合并赔付', 'difficulty': 0.85, 'min_quality': 0.82, 'max_cost': 0.06, 'sla_ms': 1000}
{'id': 'q4', 'intent': '实时聊天短答', 'difficulty': 0.3, 'min_quality': 0.68, 'max_cost': 0.008, 'sla_ms': 200}
{'id': 'q5', 'intent': '合同条款对比', 'difficulty': 0.75, 'min_quality': 0.79, 'max_cost': 0.025, 'sla_ms': 500}
{'id': 'q6', 'intent': '高风险账户申诉', 'difficulty': 0.95, 'min_quality': 0.88, 'max_cost': 0.06, 'sla_ms': 1000}
候选模型画像 {'small': {'base_quality': 0.76, 'difficulty_penalty': 0.22, 'cost': 0.003, 'latency_ms': 140}, 'medium': {'base_quality': 0.87, 'difficulty_penalty': 0.09, 'cost': 0.014, 'latency_ms': 360}, 'large': {'base_quality': 0.94, 'difficulty_penalty': 0.03, 'cost': 0.052, 'latency_ms': 820}}


## 2. Baseline（基线）：所有请求都走大模型

全量大模型通常质量较高，但简单请求浪费成本，严格延迟请求还会违反 SLO。基线与核心路由使用同一质量公式和请求集。


In [2]:
def predicted_quality12(model12, difficulty12):  # 根据离线画像估计请求质量。
    profile12 = models12[model12]  # 读取候选模型画像。
    return profile12["base_quality"] - profile12["difficulty_penalty"] * difficulty12  # 计算难度条件质量。
baseline_rows12 = []  # 收集全大模型基线结果。
for request12 in requests12:  # 对每条请求固定选择大模型。
    quality12 = predicted_quality12("large", request12["difficulty"])  # 估计大模型质量。
    feasible12 = models12["large"]["cost"] <= request12["max_cost"] and models12["large"]["latency_ms"] <= request12["sla_ms"]  # 检查成本与延迟硬约束。
    baseline_rows12.append((request12["id"], "large", round(quality12, 3), models12["large"]["cost"], models12["large"]["latency_ms"], feasible12))  # 保存基线质量和约束结果。
print("全大模型基线：id | model | quality | cost | latency | feasible")  # 输出基线表头。
for row12 in baseline_rows12:  # 逐条展示浪费或违约。
    print(row12)  # 输出一条基线路由结果。


全大模型基线：id | model | quality | cost | latency | feasible
('q1', 'large', 0.937, 0.052, 820, False)
('q2', 'large', 0.926, 0.052, 820, False)
('q3', 'large', 0.914, 0.052, 820, True)
('q4', 'large', 0.931, 0.052, 820, False)
('q5', 'large', 0.917, 0.052, 820, False)
('q6', 'large', 0.911, 0.052, 820, True)


## 3. 核心实现：先硬约束，再最大化边际效用

对每个请求计算所有模型质量，先过滤成本、延迟和最低质量不满足者。在可行集合中选择成本最低者；若没有模型可行，则返回人工兜底而不是静默降级。输出每个候选的拒绝原因。


In [3]:
def route12(request12, unavailable12=None):  # 实现质量成本延迟感知路由。
    unavailable12 = unavailable12 or set()  # 初始化当前不可用模型集合。
    decisions12 = []  # 收集每个候选的指标与拒绝原因。
    for name12, profile12 in models12.items():  # 遍历所有模型画像。
        quality12 = predicted_quality12(name12, request12["difficulty"])  # 估计当前请求质量。
        reasons12 = []  # 收集硬约束不满足原因。
        if name12 in unavailable12:  # 检查熔断或容量不可用状态。
            reasons12.append("unavailable")  # 标记模型不可选。
        if quality12 < request12["min_quality"]:  # 检查最低质量要求。
            reasons12.append("quality")  # 标记质量不足。
        if profile12["cost"] > request12["max_cost"]:  # 检查单请求成本上限。
            reasons12.append("cost")  # 标记成本超限。
        if profile12["latency_ms"] > request12["sla_ms"]:  # 检查延迟 SLO。
            reasons12.append("latency")  # 标记延迟超限。
        decisions12.append({"model": name12, "quality": round(quality12, 3), "cost": profile12["cost"], "latency": profile12["latency_ms"], "reasons": reasons12})  # 保存候选决策轨迹。
    feasible12 = [item12 for item12 in decisions12 if not item12["reasons"]]  # 提取满足全部硬约束的模型。
    choice12 = min(feasible12, key=lambda item12: (item12["cost"], -item12["quality"]))["model"] if feasible12 else "human_review"  # 选择最低成本可行模型或人工兜底。
    return choice12, decisions12  # 返回路由结果和完整解释。
routed_rows12 = []  # 收集所有请求的路由结果。
for request12 in requests12:  # 对每条业务请求运行核心路由。
    choice12, decisions12 = route12(request12)  # 获取模型选择和候选轨迹。
    routed_rows12.append((request12["id"], choice12, decisions12))  # 保存逐请求路由解释。
print("核心路由过程：id | choice | candidate decisions")  # 输出候选过滤轨迹表头。
for row12 in routed_rows12:  # 逐条展示为何选择或拒绝模型。
    print(row12)  # 输出一条完整路由解释。


核心路由过程：id | choice | candidate decisions
('q1', 'small', [{'model': 'small', 'quality': 0.738, 'cost': 0.003, 'latency': 140, 'reasons': []}, {'model': 'medium', 'quality': 0.861, 'cost': 0.014, 'latency': 360, 'reasons': ['cost', 'latency']}, {'model': 'large', 'quality': 0.937, 'cost': 0.052, 'latency': 820, 'reasons': ['cost', 'latency']}])
('q2', 'medium', [{'model': 'small', 'quality': 0.661, 'cost': 0.003, 'latency': 140, 'reasons': ['quality']}, {'model': 'medium', 'quality': 0.83, 'cost': 0.014, 'latency': 360, 'reasons': []}, {'model': 'large', 'quality': 0.926, 'cost': 0.052, 'latency': 820, 'reasons': ['cost', 'latency']}])
('q3', 'large', [{'model': 'small', 'quality': 0.573, 'cost': 0.003, 'latency': 140, 'reasons': ['quality']}, {'model': 'medium', 'quality': 0.793, 'cost': 0.014, 'latency': 360, 'reasons': ['quality']}, {'model': 'large', 'quality': 0.914, 'cost': 0.052, 'latency': 820, 'reasons': []}])
('q4', 'small', [{'model': 'small', 'quality': 0.694, 'cost': 0.003,

## 4. 结果表与结果解读

路由器让简单请求走小模型，中等请求走中模型，高难请求走大模型；若质量、成本、延迟无法同时满足则交给人工。结果同时报告平均预测质量、成本和 SLO 违约数。


In [4]:
routed_lookup12 = {row12[0]: row12[1] for row12 in routed_rows12}  # 建立请求到选择模型的映射。
routed_costs12 = []  # 收集非人工请求成本。
routed_qualities12 = []  # 收集非人工请求预测质量。
routed_violations12 = 0  # 统计核心路由硬约束违约数。
for request12 in requests12:  # 汇总逐请求路由指标。
    choice12 = routed_lookup12[request12["id"]]  # 读取当前请求选择。
    if choice12 == "human_review":  # 跳过无法自动处理的请求。
        continue  # 人工兜底不计模型成本质量。
    profile12 = models12[choice12]  # 读取选择模型画像。
    quality12 = predicted_quality12(choice12, request12["difficulty"])  # 估计实际选择质量。
    routed_costs12.append(profile12["cost"])  # 累加自动处理成本。
    routed_qualities12.append(quality12)  # 累加自动处理质量。
    routed_violations12 += int(profile12["cost"] > request12["max_cost"] or profile12["latency_ms"] > request12["sla_ms"] or quality12 < request12["min_quality"])  # 检查硬约束是否被违反。
baseline_cost12 = sum(row12[3] for row12 in baseline_rows12)  # 计算全大模型总成本。
routed_cost12 = sum(routed_costs12)  # 计算核心路由自动处理总成本。
baseline_violations12 = sum(not row12[5] for row12 in baseline_rows12)  # 统计全大模型成本或延迟违约。
print("方法 | 总模型成本 | 自动处理数 | 硬约束违约 | 平均预测质量")  # 输出路由对照表头。
print("always_large", round(baseline_cost12, 3), len(requests12), baseline_violations12, round(sum(row12[2] for row12 in baseline_rows12) / len(baseline_rows12), 3))  # 展示大模型基线。
print("constraint_router", round(routed_cost12, 3), len(routed_qualities12), routed_violations12, round(sum(routed_qualities12) / len(routed_qualities12), 3))  # 展示约束路由结果。
print("结果解读：降低成本来自按难度使用小模型，无法满足三重约束的请求必须显式兜底")  # 解释成本收益和人工路径。


方法 | 总模型成本 | 自动处理数 | 硬约束违约 | 平均预测质量
always_large 0.312 6 4 0.923
constraint_router 0.138 6 0 0.815
结果解读：降低成本来自按难度使用小模型，无法满足三重约束的请求必须显式兜底


## 5. 失败案例与修正：画像过期且中模型延迟熔断

离线画像显示 medium 为 360ms，但线上依赖抖动使其达到 760ms。若继续按旧画像路由，`q2` 会违反 500ms SLO；修正是容量监控触发熔断，在本次请求中排除 medium 并重新求解。


In [5]:
failure_request12 = next(request12 for request12 in requests12 if request12["id"] == "q2")  # 选取中模型原本可行的规则解释请求。
stale_choice12, _ = route12(failure_request12)  # 使用过期画像获取原始选择。
observed_latency12 = {"medium": 760}  # 模拟线上监控发现中模型延迟异常。
unavailable12 = {name12 for name12, latency12 in observed_latency12.items() if latency12 > failure_request12["sla_ms"]}  # 根据请求SLO触发模型熔断。
fixed_choice12, fixed_trace12 = route12(failure_request12, unavailable12)  # 排除不可用模型后重新路由。
print("失败行为：旧画像选择", stale_choice12, "实际延迟", observed_latency12.get(stale_choice12))  # 展示画像漂移造成的违约。
print("修正行为：熔断集合", unavailable12, "重新选择", fixed_choice12)  # 展示实时容量门禁后的路由。
print("修正候选轨迹", fixed_trace12)  # 输出重新求解时每个模型的拒绝原因。


失败行为：旧画像选择 medium 实际延迟 760
修正行为：熔断集合 {'medium'} 重新选择 human_review
修正候选轨迹 [{'model': 'small', 'quality': 0.661, 'cost': 0.003, 'latency': 140, 'reasons': ['quality']}, {'model': 'medium', 'quality': 0.83, 'cost': 0.014, 'latency': 360, 'reasons': ['unavailable']}, {'model': 'large', 'quality': 0.926, 'cost': 0.052, 'latency': 820, 'reasons': ['cost', 'latency']}]


## 6. 生产边界与路由制品

真实质量预测来自大规模日志和黄金集，延迟还受队列、缓存和输出长度影响。需要容量感知、熔断、探索比例、propensity 日志、预算日配额和在线回滚；教学公式不能直接用于价格或 SLA。


In [6]:
router_contract12 = {"profiles": "model-benchmark-2026-07", "quality_model": "difficulty-v3", "capacity_poll_seconds": 5, "fallback": "human_review", "propensity_logged": True}  # 定义可审计模型路由合同。
print("模型路由发布制品", router_contract12)  # 展示画像、难度模型和容量策略版本。
print("生产替换点：真实黄金集质量模型、在线队列延迟、熔断、预算配额、探索与IPS评估")  # 说明教学画像的边界。


模型路由发布制品 {'profiles': 'model-benchmark-2026-07', 'quality_model': 'difficulty-v3', 'capacity_poll_seconds': 5, 'fallback': 'human_review', 'propensity_logged': True}
生产替换点：真实黄金集质量模型、在线队列延迟、熔断、预算配额、探索与IPS评估


## 7. 最小回归测试

断言保护案例规模、成本收益、硬约束和熔断反例。


In [7]:
assert len(requests12) >= 5  # 保证路由案例覆盖不同难度和SLO。
assert routed_cost12 < baseline_cost12  # 保证约束路由降低模型成本。
assert routed_violations12 == 0  # 保证自动路由结果不违反硬约束。
assert routed_lookup12["q1"] == "small"  # 保证简单低延迟问题选择小模型。
assert stale_choice12 == "medium" and fixed_choice12 != "medium"  # 保证延迟熔断能够改变过期路由。
print("最小回归测试通过：质量成本延迟约束和容量熔断保持稳定")  # 显示路由关键性质已验证。


最小回归测试通过：质量成本延迟约束和容量熔断保持稳定
